In [2]:
import numpy as np
import h5py
import scipy.io
from sklearn import metrics
import pandas as pd
import os
os.environ['THEANO_FLAGS'] = "device=cuda0,force_device=True,floatX=float32,gpuarray.preallocate=0.3"
import theano

ERROR (theano.gpuarray): pygpu was configured but could not be imported or is too old (version 0.7 or higher required)
NoneType: None


In [3]:
from keras.layers import Embedding
from keras.models import Sequential
from keras.models import Model
from keras.layers import Dense, Dropout, Activation, Flatten, Layer, merge, Input, Concatenate, Reshape, concatenate,Lambda,multiply,Permute,Reshape,RepeatVector
from keras.layers.convolutional import Conv1D, MaxPooling1D
from keras.layers.pooling import GlobalMaxPooling1D
from keras.layers.recurrent import LSTM
from keras.layers.wrappers import Bidirectional, TimeDistributed
from keras.models import load_model
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras import optimizers
from keras import backend as K
from keras import regularizers

In [4]:
import keras;
print(keras.__version__)

2.5.0


### Load data (training and validation)

In [5]:
#data_folder = "/scratch2/yibeijia/data/train_test_data/"
data_folder = "/Users/yibeijia/Downloads/nucleosome_occupancy/data/train_test_data/"


trainmat = scipy.io.loadmat(data_folder+'Train_data.mat')
validmat = scipy.io.loadmat(data_folder+'Test_data.mat')

### need to add one more dimension to training data to represent each sequence!

X_train = np.array(trainmat['Train_data'])
y_train = np.array(trainmat['Train_labels']).T

# trainmat.close()

In [26]:
X_train.shape

(226334, 147, 4)

In [27]:
y_train.shape

(226334, 1)

### Run TBiNet

In [6]:
sequence_input = Input(shape=(147,4))

# Convolutional Layer
output = Conv1D(320,kernel_size=26,padding="valid",activation="relu")(sequence_input)
output = MaxPooling1D(pool_size=13, strides=13)(output)
output = Dropout(0.2)(output)

In [7]:
#Attention Layer
attention = Dense(1)(output)
attention = Permute((2, 1))(attention)
attention = Activation('softmax')(attention)
attention = Permute((2, 1))(attention)
attention = Lambda(lambda x: K.mean(x, axis=2), name='attention',output_shape=(75,))(attention)
attention = RepeatVector(320)(attention)
attention = Permute((2,1))(attention)
output = multiply([output, attention])

#BiLSTM Layer
output = Bidirectional(LSTM(320,return_sequences=True))(output)
output = Dropout(0.5)(output)

flat_output = Flatten()(output)

#FC Layer
FC_output = Dense(695)(flat_output)
FC_output = Activation('relu')(FC_output)

#Output Layer
# output = Dense(1)(FC_output)

output = Dense(1)(FC_output)
output = Activation('sigmoid')(output)

In [8]:
model = Model(inputs=sequence_input, outputs=output)

print('compiling model')
model.compile(loss='binary_crossentropy', optimizer='adam')

print('model summary')
model.summary()

checkpointer = ModelCheckpoint(filepath="./model/tbinet.{epoch:02d}-{val_loss:.2f}.hdf5", verbose=1, save_best_only=False)
earlystopper = EarlyStopping(monitor='val_loss', patience=10, verbose=1)

compiling model
model summary
Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 147, 4)]     0                                            
__________________________________________________________________________________________________
conv1d (Conv1D)                 (None, 122, 320)     33600       input_1[0][0]                    
__________________________________________________________________________________________________
max_pooling1d (MaxPooling1D)    (None, 9, 320)       0           conv1d[0][0]                     
__________________________________________________________________________________________________
dropout (Dropout)               (None, 9, 320)       0           max_pooling1d[0][0]              
________________________________________________________________

In [36]:
validmat['Test_data'].shape

(150890, 147, 4)

In [32]:
b=validmat['Test_labels'].T
b.shape

(150890, 1)

In [1]:
# model.fit(X_train, y_train, batch_size=100, epochs=60, shuffle=True, verbose=1, validation_data=(np.transpose(validmat['Train_data'],axes=(0,2,1)),validmat['Train_vals'][:,125:815]), callbacks=[checkpointer,earlystopper])
model.fit(X_train, y_train, batch_size=100, epochs=2, shuffle=True, verbose=1, validation_data=(validmat['Test_data'],validmat['Test_labels'].T), callbacks=[checkpointer,earlystopper])

model.save('./model/tbinet_stuff.h5')

NameError: name 'model' is not defined